# FAQ Agent

In [24]:
from openai import OpenAI
import requests
from minsearch import AppendableIndex

from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.runners import DisplayingRunnerCallback
from toyaikit.tools import Tools

### Refactored Agent

In [25]:
openai_client = OpenAI()

In [26]:
def download_docs():
    docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
    docs_response = requests.get(docs_url)
    documents_raw = docs_response.json()

    documents = []

    for course in documents_raw:
        course_name = course['course']

        for doc in course['documents']:
            doc['course'] = course_name
            documents.append(doc)

    return documents

In [27]:
documents = download_docs()

In [28]:
index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [29]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
    )

    return results

In [30]:
# describe the search tool available for the agent
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [31]:
instructions = """
You're a course teaching assistant. 
You're given a question from a course student and your task is to answer it.

If you want to look up the answer, explain why before making the call
""".strip()

In [32]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [33]:
chat_interface = IPythonChatInterface()

In [34]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient()
)

In [35]:
callback = DisplayingRunnerCallback(chat_interface)

question = 'how do I install kafka'
loop_result = runner.loop(prompt=question, callback=callback)

In [ ]:
runner.run();

Chat ended.


LoopResult(new_messages=[{'role': 'developer', 'content': "You're a course teaching assistant. \nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up the answer, explain why before making the call"}, {'role': 'user', 'content': ''}, ResponseOutputMessage(id='msg_0dc396b7536fb3100068ff811c010081a09e8cf453c160ca0a', content=[ResponseOutputText(annotations=[], text="It seems you didn't include your question. Please provide it so I can assist you!", type='output_text', logprobs=[])], role='assistant', status='completed', type='message'), {'role': 'user', 'content': 'I just discovered the course. Can I still enroll?'}, ResponseOutputMessage(id='msg_0dc396b7536fb3100068ff812b1e1081a09c855053d5b59a90', content=[ResponseOutputText(annotations=[], text="It's not clear whether students can enroll late in the course. To provide the most accurate information, I will check the course FAQ for any specific details regarding late enrollment. Let's see 

### Multiple tools

In [37]:
# create a new tool for adding documentation to the search engine
def add_entry(question, answer):
    doc = {
        'question': question,
        'text': answer,
        'section': 'user added',
        'course': 'data-engineering-zoomcamp'
    }
    index.append(doc)

In [38]:
add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add an entry to the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question to be added to the FAQ database",
            },
            "answer": {
                "type": "string",
                "description": "The answer to the question",
            }
        },
        "required": ["question", "answer"],
        "additionalProperties": False
    }
}

In [39]:
agent_tools.add_tool(add_entry, add_entry_tool)

In [40]:
runner.run();

Chat ended.
